# Clean Divvy Trip Data

This notebook loads raw Divvy trip data, combines monthly files, creates time-based features, removes invalid trip durations, and saves a cleaned trip-level dataset for downstream analysis.

In [15]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("D:/Projects/chicago-bike-demand")
RAW_DIR = BASE_DIR / "01_data" / "Raw"
PROCESSED_DIR = BASE_DIR / "01_data" / "Processed"

print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Base directory: D:\Projects\chicago-bike-demand
Raw directory: D:\Projects\chicago-bike-demand\01_data\Raw
Processed directory: D:\Projects\chicago-bike-demand\01_data\Processed


### Load Raw Files

In [16]:
files = sorted(RAW_DIR.glob("*.csv"))

print(f"Found {len(files)} raw files")

df_list = []

for file in files:
    temp_df = pd.read_csv(file)
    print(f"Reading: {file.name} | Rows: {len(temp_df):,}")
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

print("\nCombined dataset shape:", df.shape)
df.head()

Found 3 raw files
Reading: 202503-divvy-tripdata.csv | Rows: 298,155
Reading: 202504-divvy-tripdata.csv | Rows: 371,341
Reading: 202505-divvy-tripdata.csv | Rows: 502,456

Combined dataset shape: (1171952, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,NaN,NaN,Canal St & Jackson Blvd,13138,41.91,-87.67,41.878125,-87.639968,member
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,NaN,NaN,Canal St & Jackson Blvd,13138,41.87,-87.63,41.878125,-87.639968,member
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.914027,-87.705126,casual


### Inspect Columns and Missing Values

In [17]:
print("Columns:")
print(df.columns.tolist())

print("\nRaw shape:")
print(df.shape)

print("\nMissing values by column:")
print(df.isna().sum())

Columns:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

Raw shape:
(1171952, 13)

Missing values by column:
ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name    229893
start_station_id      229893
end_station_name      239055
end_station_id        239055
start_lat                  0
start_lng                  0
end_lat                 1081
end_lng                 1081
member_casual              0
dtype: int64


## Convert timestamps

In [18]:
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")

print("Missing started_at after conversion:", df["started_at"].isna().sum())
print("Missing ended_at after conversion:", df["ended_at"].isna().sum())

Missing started_at after conversion: 0
Missing ended_at after conversion: 0


## Create trip duration

In [19]:
df["trip_duration_min"] = (
    df["ended_at"] - df["started_at"]
).dt.total_seconds() / 60

print(df["trip_duration_min"].describe())

count    1.171952e+06
mean     1.512774e+01
std      5.300939e+01
min      1.100000e-03
25%      5.087117e+00
50%      8.890517e+00
75%      1.567822e+01
max      1.559918e+03
Name: trip_duration_min, dtype: float64


## Create time-based features

In [20]:
df["date"] = df["started_at"].dt.date
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.dayofweek
df["month"] = df["started_at"].dt.month

df[["started_at", "date", "hour", "day_of_week", "month"]].head()

,started_at,date,hour,day_of_week,month
0,2025-03-18 08:39:20.065,2025-03-18,8,1,3
1,2025-03-24 16:04:22.239,2025-03-24,16,0,3
2,2025-03-10 16:06:19.708,2025-03-10,16,0,3
3,2025-03-21 14:28:14.579,2025-03-21,14,4,3
4,2025-03-14 17:54:14.484,2025-03-14,17,4,3


## Create behavioral flags

### Create Weekend and Rush Hour Flags

In [22]:
df["is_weekend"] = df["day_of_week"].isin([5, 6])
df["rush_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18])

df[["day_of_week", "hour", "is_weekend", "rush_hour"]].head()

,day_of_week,hour,is_weekend,rush_hour
0,1,8,False,True
1,0,16,False,True
2,0,16,False,True
3,4,14,False,False
4,4,17,False,True


## Clean invalid trip durations
Trips with durations less than or equal to 0 were removed as invalid.  
Trips over 180 minutes were treated as extreme outliers and removed to reduce distortion in downstream analysis.

In [23]:
initial_rows = len(df)

df = df[df["trip_duration_min"] > 0]
df = df[df["trip_duration_min"] < 180]

final_rows = len(df)

print("Initial rows:", f"{initial_rows:,}")
print("Final rows:", f"{final_rows:,}")
print("Rows removed:", f"{initial_rows - final_rows:,}")

Initial rows: 1,171,952
Final rows: 1,168,919
Rows removed: 3,033


## Validate cleaned data

In [24]:
print("Final shape:", df.shape)
print("Max trip duration:", df["trip_duration_min"].max())
print("Duplicate ride_id count:", df["ride_id"].duplicated().sum())

print("\nMissing values by column:")
print(df.isna().sum())

Final shape: (1168919, 20)
Max trip duration: 179.97003333333333
Duplicate ride_id count: 0

Missing values by column:
ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name    229683
start_station_id      229683
end_station_name      237683
end_station_id        237683
start_lat                  0
start_lng                  0
end_lat                    7
end_lng                    7
member_casual              0
trip_duration_min          0
date                       0
hour                       0
day_of_week                0
month                      0
is_weekend                 0
rush_hour                  0
dtype: int64


## Missing values assessment

Missing values remain in station-related fields such as `start_station_name`, `start_station_id`, `end_station_name`, and `end_station_id`.

These null values were retained because this project focuses on city-level demand aggregation rather than station-level analysis.

Since downstream steps aggregate trips by date and hour, these station-level nulls do not affect the final demand calculations.

## Save cleaned trip data

In [25]:
output_path = PROCESSED_DIR / "cleaned_divvy_trips.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("Cleaned data saved")
print(output_path)

Cleaned data saved
D:\Projects\chicago-bike-demand\01_data\Processed\cleaned_divvy_trips.csv


## Final output check

In [26]:
check_df = pd.read_csv(output_path)

print("Saved file shape:", check_df.shape)
print("Saved max trip duration:", check_df["trip_duration_min"].max())
check_df.head()

Saved file shape: (1168919, 20)
Saved max trip duration: 179.97003333333333


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,trip_duration_min,date,hour,day_of_week,month,is_weekend,rush_hour
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,NaN,NaN,Canal St & Jackson Blvd,13138,41.91,-87.67,41.878125,-87.639968,member,12.292800,2025-03-18,8,1,3,False,True
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member,23.318467,2025-03-24,16,0,3,False,True
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member,22.962483,2025-03-10,16,0,3,False,True
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,NaN,NaN,Canal St & Jackson Blvd,13138,41.87,-87.63,41.878125,-87.639968,member,6.859683,2025-03-21,14,4,3,False,False
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.914027,-87.705126,casual,23.646167,2025-03-14,17,4,3,False,True


In [27]:
print(df["trip_duration_min"].max())
print(df["ride_id"].duplicated().sum())
print(output_path.exists())

179.97003333333333
0
True
